# GIAI ĐOẠN 3: GÁN NHÃN & CHIA TẬP DỮ LIỆU

Nhãn theo số sao có sẵn khi cào nhưng bị nhiễu (nhiều review 3 sao thực chất đang chê), nên nhóm gán lại theo **nội dung chữ**:

| File | Nội dung | Dùng để |
|---|---|---|
| `data/labels/human_labels.xlsx` | Nhóm gán tay 500 câu | Tập **test** chuẩn |
| `data/labels/ai_labels.csv` | Claude gán **toàn bộ** câu theo `prompt_ai_label1.md` (luật rút từ cách nhóm gán tay) | Nhãn **train** và **validation** |
| `data/labels/ai_label1.xlsx` | Nhãn AI cho 500 câu trên, kèm các câu lệch với người | Tham khảo |

**Đầu ra:** `data/splits/train.csv`, `validation.csv`, `test.csv`, mỗi file 2 cột `text`, `label` (`-1` tiêu cực, `0` trung tính, `1` tích cực).

### Bước 1: Đọc dữ liệu sạch và các file nhãn

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

DATA_DIR  = Path("../data")
SPLIT_DIR = DATA_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
LABELS, NAMES = [-1, 0, 1], ["Tiêu cực", "Trung tính", "Tích cực"]

# Dữ liệu đã làm sạch. nhan_sao là nhãn suy từ số sao lúc cào, chỉ dùng để so sánh.
df = pd.read_csv(DATA_DIR / "processed" / "reviews_clean.csv")
df["nhan_sao"] = df["nhom_cam_xuc"].map(dict(zip(NAMES, LABELS)))

# Nhãn AI cho toàn bộ câu
ai = pd.read_csv(DATA_DIR / "labels" / "ai_labels.csv").set_index("review_id")["label"]
assert ai.index.is_unique and set(ai.index) == set(df["review_id"]), \
    "ai_labels.csv phải có đủ mọi câu của reviews_clean.csv (cào lại dữ liệu thì phải gán lại)"
assert ai.isin(LABELS).all(), "Nhãn chỉ được là -1, 0, 1"
df["label"] = df["review_id"].map(ai)

# Nhãn người cho 500 câu
nguoi = pd.read_excel(DATA_DIR / "labels" / "human_labels.xlsx").set_index("review_id")["label"]
assert nguoi.notna().all(), f"human_labels.xlsx còn {nguoi.isna().sum()} câu chưa gán nhãn"
nguoi = nguoi.astype(int)
assert nguoi.isin(LABELS).all() and nguoi.index.isin(df["review_id"]).all()

print(f"{len(df):,} câu | nhãn AI: {len(ai):,} câu | nhãn người: {len(nguoi)} câu")

### Bước 2: Chất lượng nhãn: so với nhãn người (tỉ lệ khớp và Cohen's Kappa)

In [ ]:
sao = df.set_index("review_id")["nhan_sao"]

def so_voi_nguoi(nguon, nhan):
    nhan = nhan[nguoi.index]
    return {"Nguồn nhãn": nguon, "Tỉ lệ khớp": (nhan == nguoi).mean(),
            "Cohen's Kappa": cohen_kappa_score(nguoi, nhan)}

display(pd.DataFrame([so_voi_nguoi("Số sao", sao), so_voi_nguoi("AI (Claude)", ai)]).round(3))
print("Lưu ý: luật trong prompt rút ra từ chính các câu này, nên độ khớp của nhãn AI ở đây cao hơn thực tế.")

### Bước 3: Chia tập Train / Validation / Test
- **Test**: 500 câu nhóm gán tay, dùng **nhãn người**
- **Validation**: 15% tổng số câu, chia phân tầng theo nhãn, dùng **nhãn AI**
- **Train**: phần còn lại, dùng **nhãn AI**

Mỗi câu chỉ nằm trong một tập.

In [ ]:
la_test = df["review_id"].isin(nguoi.index)
test = df[la_test].assign(label=lambda d: d["review_id"].map(nguoi))
train, val = train_test_split(df[~la_test], test_size=round(0.15 * len(df)),
                              stratify=df.loc[~la_test, "label"], random_state=SEED)

for ten, tap in [("train", train), ("validation", val), ("test", test)]:
    tap.rename(columns={"noi_dung": "text"})[["text", "label"]].to_csv(
        SPLIT_DIR / f"{ten}.csv", index=False, encoding="utf-8-sig")

bang = pd.DataFrame({ten: tap["label"].value_counts().reindex(LABELS)
                     for ten, tap in [("Train", train), ("Validation", val), ("Test", test)]}).set_axis(NAMES)
bang.loc["Tổng"] = bang.sum()
print(f"Đã lưu 3 tập vào {SPLIT_DIR.resolve()}")
bang